# Phase 2: High-Dimensional Dictionary Learning (KSAE Training)

In this phase, we train a Sparse Autoencoder (SAE) on the hidden states of the selected layer $L^*$. We use a **k-Sparse Autoencoder (k-SAE)**, which forces exactly $k$ neurons to fire for any given input, ensuring high interpretability and feature isolation.

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# Phase 2: k-SAE Training on SD 2.1 Text Encoder (Layer 20)
# ══════════════════════════════════════════════════════════════════════════════

import os
import sys
import subprocess
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from transformers import CLIPTokenizer, CLIPTextModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ── Configuration ─────────────────────────────────────────────────────────────
WORKSPACE_DIR = "workspace"
L_STAR = 20              # Target text encoder layer from Phase 1
D_MODEL = 1024           # SD 2.1 text encoder hidden dim
N_DIRS = 4096            # SAE expansion (4× overcomplete)
K = 32                   # Top-k sparsity
EPOCHS = 5               # Reduced from 10 since corpus is 20× larger
BATCH_SIZE = 4096
LR = 3e-4

MODEL_ID = "sd2-community/stable-diffusion-2-1"
CORPUS_PATH = f"{WORKSPACE_DIR}/datasets/sae_train_corpus/train_prompts.txt"
ACTS_PATH = f"{WORKSPACE_DIR}/activations/sae_train_acts_layer{L_STAR}.pt"
CHECKPOINT_DIR = f"{WORKSPACE_DIR}/models/sae_checkpoint_layer{L_STAR}"

os.makedirs(f"{WORKSPACE_DIR}/datasets/sae_train_corpus", exist_ok=True)
os.makedirs(f"{WORKSPACE_DIR}/activations", exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Training k-SAE: {D_MODEL}→{N_DIRS}, k={K}, {EPOCHS} epochs")

2026-04-29 05:27:02.439659: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777440422.457593   32328 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777440422.463590   32328 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-29 05:27:02.486454: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Device: cuda
Training k-SAE: 1024→4096, k=32, 5 epochs


In [2]:
# ── Clone sdxl-unbox for SAE implementation ───────────────────────────────────
SDXL_UNBOX_DIR = f"{WORKSPACE_DIR}/sdxl_unbox"

if not os.path.exists(SDXL_UNBOX_DIR):
    subprocess.check_call([
        "git", "clone",
        "https://github.com/surkovv/sdxl-unbox.git",
        SDXL_UNBOX_DIR
    ])
    print("✅ Cloned sdxl-unbox")
else:
    print("✅ sdxl-unbox already present")

if SDXL_UNBOX_DIR not in sys.path:
    sys.path.insert(0, SDXL_UNBOX_DIR)

from SAE.sae import SparseAutoencoder

print("✅ Imported SparseAutoencoder from sdxl-unbox")

✅ sdxl-unbox already present
✅ Imported SparseAutoencoder from sdxl-unbox


In [3]:
# ── Download 500k CC3M captions ───────────────────────────────────────────────
from datasets import load_dataset

TARGET_SIZE = 500_000

if os.path.exists(CORPUS_PATH):
    with open(CORPUS_PATH) as f:
        training_prompts = [line.strip() for line in f if line.strip()]
    print(f"✅ Loaded existing corpus: {len(training_prompts):,} prompts")
else:
    print(f"Downloading CC3M captions (target: {TARGET_SIZE:,})...")
    ds = load_dataset(
        "google-research-datasets/conceptual_captions",
        split="train",
        streaming=True,
        trust_remote_code=True,
    )
    
    training_prompts = []
    for ex in tqdm(ds, total=TARGET_SIZE, desc="Downloading"):
        cap = ex.get("caption", "").strip()
        if cap:
            training_prompts.append(cap)
        if len(training_prompts) >= TARGET_SIZE:
            break
    
    with open(CORPUS_PATH, "w") as f:
        f.write("\n".join(training_prompts))
    print(f"✅ Saved {len(training_prompts):,} prompts → {CORPUS_PATH}")

print(f"Corpus size: {len(training_prompts):,}")

✅ Loaded existing corpus: 500,000 prompts
Corpus size: 500,000


In [4]:
# ── Load text encoder and extract Layer 20 activations ────────────────────────
tokenizer = CLIPTokenizer.from_pretrained(
    MODEL_ID, subfolder="tokenizer", local_files_only=True
)
text_encoder = CLIPTextModel.from_pretrained(
    MODEL_ID, subfolder="text_encoder", local_files_only=True
).to(device)
text_encoder.eval()

print(f"Text encoder has {len(text_encoder.text_model.encoder.layers)} layers")

def extract_activations(prompts, batch_size=128):
    """
    Extract Layer L_STAR hidden states from text encoder.
    Returns: (N, 1024) tensor of mean-pooled activations.
    """
    all_acts = []
    with torch.no_grad():
        for i in tqdm(range(0, len(prompts), batch_size), desc="Extracting"):
            batch = prompts[i:i+batch_size]
            inputs = tokenizer(
                batch,
                padding="max_length",
                max_length=tokenizer.model_max_length,
                truncation=True,
                return_tensors="pt"
            ).to(device)
            
            outputs = text_encoder(**inputs, output_hidden_states=True)
            # hidden_states is tuple of length 24 (embedding + 23 layers)
            # L_STAR=20 corresponds to index 21
            layer_acts = outputs.hidden_states[L_STAR + 1]  # (B, seq_len, 1024)
            mean_acts = layer_acts.mean(dim=1)              # (B, 1024)
            all_acts.append(mean_acts.cpu())
    
    return torch.cat(all_acts, dim=0)

# Cache activations to avoid re-extraction on kernel restart
if os.path.exists(ACTS_PATH):
    print(f"✅ Loading cached activations from {ACTS_PATH}")
    train_activations = torch.load(ACTS_PATH, map_location="cpu")
else:
    print(f"Extracting activations for {len(training_prompts):,} prompts...")
    train_activations = extract_activations(training_prompts)
    torch.save(train_activations, ACTS_PATH)
    print(f"✅ Saved activations → {ACTS_PATH}")

print(f"Activations shape: {train_activations.shape}")
# Expected: torch.Size([500000, 1024])

Text encoder has 23 layers
✅ Loading cached activations from workspace/activations/sae_train_acts_layer20.pt
Activations shape: torch.Size([500000, 1024])


In [5]:
# ── Initialize k-SAE ──────────────────────────────────────────────────────────
sae = SparseAutoencoder(
    n_dirs_local=N_DIRS,
    d_model=D_MODEL,
    k=K,
    auxk=None,
    dead_steps_threshold=0,
).to(device)

optimizer = torch.optim.Adam(sae.parameters(), lr=LR)

# Create DataLoader
dataset = TensorDataset(train_activations)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"✅ SAE initialized: {D_MODEL}→{N_DIRS}, k={K}")
print(f"Total batches per epoch: {len(dataloader):,}")

✅ SAE initialized: 1024→4096, k=32
Total batches per epoch: 123


In [6]:
# ── Training loop ─────────────────────────────────────────────────────────────
sae.train()
losses = []

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    progress = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for (batch_acts,) in progress:
        batch_acts = batch_acts.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        recon_acts, hidden = sae(batch_acts)
        
        # Reconstruction loss
        loss = F.mse_loss(recon_acts, batch_acts)
        
        # Backward
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        progress.set_postfix({"loss": f"{loss.item():.6f}"})
    
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS} | Avg Loss: {avg_loss:.6f}")

print("✅ Training complete")

Epoch 1/5: 100%|██████████| 123/123 [00:05<00:00, 22.10it/s, loss=1.386944]


Epoch 1/5 | Avg Loss: 5.150885


Epoch 2/5: 100%|██████████| 123/123 [00:05<00:00, 24.15it/s, loss=0.111526]


Epoch 2/5 | Avg Loss: 0.326981


Epoch 3/5: 100%|██████████| 123/123 [00:05<00:00, 22.04it/s, loss=0.101055]


Epoch 3/5 | Avg Loss: 0.107475


Epoch 4/5: 100%|██████████| 123/123 [00:05<00:00, 22.86it/s, loss=0.099362]


Epoch 4/5 | Avg Loss: 0.101600


Epoch 5/5: 100%|██████████| 123/123 [00:04<00:00, 26.90it/s, loss=0.092302]

Epoch 5/5 | Avg Loss: 0.095932
✅ Training complete


In [7]:
# ── Save trained SAE checkpoint ───────────────────────────────────────────────
checkpoint = {
    "model_state_dict": sae.state_dict(),
    "n_training_prompts": len(training_prompts),
    "final_loss": losses[-1],
    "config": {
        "d_model": D_MODEL,
        "n_dirs": N_DIRS,
        "k": K,
        "layer": L_STAR,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
    }
}

checkpoint_path = f"{CHECKPOINT_DIR}/ksae.pt"
torch.save(checkpoint, checkpoint_path)

print(f"✅ Checkpoint saved → {checkpoint_path}")
print(f"   Prompts: {len(training_prompts):,}")
print(f"   Final loss: {losses[-1]:.6f}")

✅ Checkpoint saved → workspace/models/sae_checkpoint_layer20/ksae.pt
   Prompts: 500,000
   Final loss: 0.095932
